In [ ]:
import os
import sys
import math
import pathlib
import numpy as np
import matplotlib.pyplot as plt


from tqdm import tqdm

import mindspore as ms
from mindspore import  ops
from mindquantum import  Simulator
from mindquantum.algorithm.library import amplitude_encoder

project_path = pathlib.Path.cwd().parent.parent
sys.path.append(f"{project_path}/src")

from datasets_utils.dataset import get_dataloader_with_idx


In [2]:
n_qubits = 8
n_layers = 0
n_train_samples = 20000
n_test_samples = 10000
batch_size = 200
data_type = 'mnist'

In [3]:
train_loader_class_dict = {}
for i in range(10):
    train_data_loader_class, test_data_loader_class = get_dataloader_with_idx(n_qubits=n_qubits, n_layers=n_layers,n_train_samples=n_train_samples, n_test_samples=n_test_samples, batch_size=batch_size, data_type=data_type, class_idx=i)
    train_loader_class_dict[i] = train_data_loader_class

In [5]:
train_data_class_dict = {i: [] for i in range(10)}
for i in range(10):
    for data,label in train_loader_class_dict[i]:
        for x in data:
            train_data_class_dict[i].append(x)

In [13]:

def AmplitudeEncoder(x):
    sim = Simulator('mqvector', n_qubits)
    encoder, parameterResolver  = amplitude_encoder(x.asnumpy(), n_qubits)
    sim.apply_circuit(encoder,parameterResolver)
    state = sim.get_qs()
    state = ms.Tensor(state, dtype=ms.complex64)
    state = state.reshape(-1, 1)  # Reshape to column vector
    rho = ops.matmul(state, ops.conj(state.T))
    return rho

In [14]:
train_class_embeddings = {}
for i in range(10):
    count = 0
    rho = np.zeros((2**n_qubits, 2**n_qubits), dtype=np.complex128)
    for data in tqdm(train_data_class_dict[i],desc=f"Processing data for class {i}",leave=False):
        rho += AmplitudeEncoder(data)
        count += 1
    train_class_embeddings[i] = rho / count


In [17]:
def trace_distance(rho, sigma):
    """Calculate the trace distance between two density matrices"""
    diff = rho - sigma
    eigvals = np.abs(np.linalg.eigvalsh(diff))
    # Calculate trace and divide by 2
    return np.sum(eigvals, axis=-1) / 2

In [19]:
# Try all possible combinations of 6 digits to find the set with maximum minimum trace distance
from itertools import combinations
from tqdm import tqdm


max_min_distance = 0
best_group = None

# Get all possible combinations of 6 digits
all_digits = list(range(10))
for group in tqdm(combinations(all_digits, 6), desc=f"Processing combinations: {len(list(combinations(all_digits, 6)))} total", leave=True):
    # Calculate trace distance between all pairs in the group
    min_distance = float('inf')
    
    # Check all pairs within the group
    for i, digit1 in enumerate(group):
        for digit2 in group[i+1:]:
            # Calculate trace distance between the two digits
            distance = trace_distance(
                train_class_embeddings[digit1], 
                train_class_embeddings[digit2]
            )
            
            # Update minimum distance for this group
            if distance < min_distance:
                min_distance = distance
    
    # Update if this group has a better minimum distance
    if min_distance > max_min_distance:
        max_min_distance = min_distance
        best_group = group

print(f"Maximum minimum trace distance: {max_min_distance}")
print(f"Best group of 6 digits: {best_group}")


Processing combinations: 210 total: 210it [00:22,  9.35it/s]

Maximum minimum trace distance: 0.5593727022280954
Best group of 6 digits: (0, 1, 3, 4, 6, 7)


In [26]:
# Partition (0, 1, 3, 4, 6, 7) into two groups of three digits each, maximizing the trace distance between the two groups
import numpy as np
from itertools import combinations

# Selected 6 digits
selected_digits = (0, 1, 3, 4, 6, 7)

max_distance = 0
best_partition = None

# Get all possible combinations of 3 digits
for group1 in combinations(selected_digits, 3):
    # Calculate the second group (remaining 3 digits)
    group2 = tuple(digit for digit in selected_digits if digit not in group1)
    
    # Calculate the average density matrices of the two groups
    avg_density1 = sum(train_class_embeddings[digit] for digit in group1) / 3
    avg_density2 = sum(train_class_embeddings[digit] for digit in group2) / 3
    
    # Calculate the trace distance between the two groups
    distance = trace_distance(avg_density1, avg_density2)
    
    # Update maximum distance
    if distance > max_distance:
        max_distance = distance
        best_partition = (group1, group2)

print(f"Maximum trace distance: {max_distance}")
print(f"Best partition: {best_partition[0]} and {best_partition[1]}")


Maximum trace distance: 0.5672352307791606
Best partition: (0, 3, 6) and (1, 4, 7)


In [27]:
# Partition (0, 3, 6) into two groups, one with 1 digit and the other with 2 digits, maximizing the trace distance
import numpy as np
from itertools import combinations

# Selected 3 digits
selected_digits = (0, 3, 6)

max_distance = 0
best_partition = None

# Get all possible combinations of 1 digit (i.e., single digit)
for group1 in combinations(selected_digits, 1):
    # Calculate the second group (remaining 2 digits)
    group2 = tuple(digit for digit in selected_digits if digit not in group1)
    
    # Calculate the average density matrices of the two groups
    avg_density1 = train_class_embeddings[group1[0]]  # Density matrix of a single digit
    avg_density2 = sum(train_class_embeddings[digit] for digit in group2) / 2  # Average density matrix of two digits
    
    # Calculate the trace distance between the two groups
    distance = trace_distance(avg_density1, avg_density2)
    
    # Update maximum distance
    if distance > max_distance:
        max_distance = distance
        best_partition = (group1, group2)

print(f"Maximum trace distance: {max_distance}")
print(f"Best partition: {best_partition[0]} and {best_partition[1]}")


Maximum trace distance: 0.5892036059221534
Best partition: (3,) and (0, 6)


In [28]:
# Partition (1, 4, 7) into two groups, one with 1 digit and the other with 2 digits, maximizing the trace distance
import numpy as np
from itertools import combinations

# Selected 3 digits
selected_digits = (1, 4, 7)

max_distance = 0
best_partition = None

# Get all possible combinations of 1 digit (i.e., single digit)
for group1 in combinations(selected_digits, 1):
    # Calculate the second group (remaining 2 digits)
    group2 = tuple(digit for digit in selected_digits if digit not in group1)
    
    # Calculate the average density matrices of the two groups
    avg_density1 = train_class_embeddings[group1[0]]  # Density matrix of a single digit
    avg_density2 = sum(train_class_embeddings[digit] for digit in group2) / 2  # Average density matrix of two digits
    
    # Calculate the trace distance between the two groups
    distance = trace_distance(avg_density1, avg_density2)
    
    # Update maximum distance
    if distance > max_distance:
        max_distance = distance
        best_partition = (group1, group2)

print(f"Maximum trace distance: {max_distance}")
print(f"Best partition: {best_partition[0]} and {best_partition[1]}")


Maximum trace distance: 0.7366496354543868
Best partition: (1,) and (4, 7)


In [29]:
# Measure the trace distance between digits 0 and 6
import numpy as np

# Select the two digits to compare
digit1 = 0
digit2 = 6

# Get the density matrices of the two digits
density1 = train_class_embeddings[digit1]
density2 = train_class_embeddings[digit2]

# Calculate the trace distance between them
distance = trace_distance(density1, density2)

print(f"Trace distance between digits {digit1} and {digit2}: {distance}")


Trace distance between digits 0 and 6: 0.6109325427671062


In [30]:
# Measure the trace distance between digits 4 and 7
import numpy as np

# Select the two digits to compare
digit1 = 4
digit2 = 7

# Get the density matrices of the two digits
density1 = train_class_embeddings[digit1]
density2 = train_class_embeddings[digit2]

# Calculate the trace distance between them
distance = trace_distance(density1, density2)

print(f"Trace distance between digits {digit1} and {digit2}: {distance}")


Trace distance between digits 4 and 7: 0.5593727022280954
